In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from urllib.parse import urlparse

# 1. CHARGEMENT DES DONNÉES DE BASE (On a besoin de l'URL pour le domaine)
df_base = pd.read_csv('../data/archive (2)/new_data_urls.csv')

# --- VÉRIFICATION 1 : LES DOUBLONS ---
print(f"Nombre de lignes AVANT nettoyage : {len(df_base)}")
df_base = df_base.drop_duplicates(subset=['url']) # On supprime les URL identiques
print(f"Nombre de lignes APRÈS nettoyage : {len(df_base)}")

# --- PRÉPARATION POUR LE GROUPE (Extraction du domaine) ---
# On crée une colonne "domaine" temporaire
def get_domain(url):
    try:
        if not url.startswith('http'): url = 'http://' + url
        return urlparse(url).netloc
    except: return "inconnu"

df_base['domaine'] = df_base['url'].apply(get_domain)

# --- VÉRIFICATION 3 : GROUP SHUFFLE SPLIT (Séparation par domaine) ---
# On sépare 80% / 20% en s'assurant que les domaines ne se croisent pas
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_base, groups=df_base['domaine']))

df_train = df_base.iloc[train_idx]
df_test = df_base.iloc[test_idx]

print(f"\nAprès séparation par domaine :")
print(f"Entraînement : {len(df_train)} URL")
print(f"Test : {len(df_test)} URL")

# --- VÉRIFICATION 2 : L'ÉQUILIBRE (Stratify) ---
# On vérifie que les proportions sont les mêmes
print("\nÉquilibre des classes (0=Phishing, 1=Sain) :")
print("Entraînement :\n", df_train['status'].value_counts(normalize=True) * 100)
print("Test :\n", df_test['status'].value_counts(normalize=True) * 100)

# 💡 NOTE : Pour la suite (l'entraînement), tu devras recalculer les features de texte 
# sur df_train et df_test séparément, ou utiliser ton fichier dataset_features.csv 
# si tu es sûre qu'il n'y a pas de doublons.

Nombre de lignes AVANT nettoyage : 822010
Nombre de lignes APRÈS nettoyage : 808042

Après séparation par domaine :
Entraînement : 642630 URL
Test : 165412 URL

Équilibre des classes (0=Phishing, 1=Sain) :
Entraînement :
 status
1    52.4837
0    47.5163
Name: proportion, dtype: float64
Test :
 status
1    54.259667
0    45.740333
Name: proportion, dtype: float64


In [2]:
import sys
sys.path.append('../src')
from features import *
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. On calcule les features de texte sur les données PROPRES
def apply_features(df):
    df['longueur'] = df['url'].apply(url_length)
    df['nb_points'] = df['url'].apply(count_dots)
    df['nb_tirets'] = df['url'].apply(count_hyphens)
    df['nb_chiffres'] = df['url'].apply(count_digits)
    df['nb_arobase'] = df['url'].apply(count_at_symbols)
    df['a_ip'] = df['url'].apply(has_ip_address)
    df['a_https'] = df['url'].apply(has_https)
    df['nb_sous_domaines'] = df['url'].apply(count_subdomains)
    df['mots_suspects'] = df['url'].apply(has_suspicious_words)
    return df

print("Calcul des features sur les données propres...")
df_train_feat = apply_features(df_train.copy())
df_test_feat = apply_features(df_test.copy())

# 2. On prépare X et y (on supprime l'url, le status et le domaine temporaire)
X_train = df_train_feat.drop(columns=['url', 'status', 'domaine'])
y_train = df_train_feat['status']

X_test = df_test_feat.drop(columns=['url', 'status', 'domaine'])
y_test = df_test_feat['status']

# 3. Entraînement
print("Entraînement du Random Forest (sans triche)...")
start = time.time()

model_clean = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_clean.fit(X_train, y_train)

print(f"Temps d'entraînement : {time.time() - start:.2f} secondes")

# 4. Évaluation
y_pred = model_clean.predict(X_test)

print(f"\nNouvelle Précision globale : {accuracy_score(y_test, y_pred) * 100:.2f} %")
print("\n--- Rapport détaillé ---")
print(classification_report(y_test, y_pred, target_names=["Phishing (0)", "Sain (1)"]))

Calcul des features sur les données propres...
Entraînement du Random Forest (sans triche)...
Temps d'entraînement : 21.72 secondes

Nouvelle Précision globale : 79.39 %

--- Rapport détaillé ---
              precision    recall  f1-score   support

Phishing (0)       0.79      0.75      0.77     75660
    Sain (1)       0.80      0.83      0.81     89752

    accuracy                           0.79    165412
   macro avg       0.79      0.79      0.79    165412
weighted avg       0.79      0.79      0.79    165412



In [5]:
import joblib

# 9. On sauvegarde le modèle dans un fichier
joblib.dump(model, '../src/model_phishing.pkl')
print("Modèle sauvegardé dans src/model_phishing.pkl avec succès !")

Modèle sauvegardé dans src/model_phishing.pkl avec succès !
